# Експорт YOLOv8n в TensorRT (FP32 / FP16 / INT8)
Запускати **на цільовому комп'ютері з NVIDIA GPU** (CUDA + TensorRT). Engine-файл прив'язаний до архітектури GPU — зібране на одній карті не завжди запуститься на іншій.

Вимоги на цільовій машині:
- NVIDIA GPU, драйвер + CUDA
- `pip install ultralytics onnx` + TensorRT від NVIDIA під свою CUDA
- цей зошит лежить поруч з `1)_YOLOv8n_Hugging_Face_TomSmaildrone-yolo-v1.pt`

In [21]:
# 0. Перевірка середовища
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Нема CUDA — TensorRT engine тут не збереться')

try:
    import tensorrt
    print('tensorrt:', tensorrt.__version__)
except ImportError:
    raise RuntimeError('Нема tensorrt — встанови пакет TensorRT від NVIDIA під свою CUDA')

torch: 2.9.0+cu128
cuda: True
gpu: NVIDIA GeForce RTX 4070 Laptop GPU
tensorrt: 11.3.0.99


In [22]:
!pip install tensorrt

In [23]:
# 2. TensorRT FP32 -> <model>_tensor_rt.engine
from pathlib import Path
from ultralytics import YOLO

SRC = Path('YOLOv8n_HuggingFace_TomSmaildrone-yolo-v1.pt')
IMGSZ = 640
DEVICE = 0
WORKSPACE = 0  # TensorRT workspace size flag; use integer MB-like value or leave unset

if ')_' in SRC.stem:
    prefix, rest = SRC.stem.split(')_', 1)
    prefix += ')'
    dst_name = f'{prefix}_tensor_rt_{rest}.engine'
else:
    dst_name = f'{SRC.stem}_tensor_rt.engine'

out = YOLO(str(SRC)).export(format='engine', imgsz=IMGSZ, device=DEVICE, workspace=WORKSPACE, verbose=False)
out = Path(out)
dst = out.parent / dst_name

if dst.exists():
    dst.unlink()

out.rename(dst)
print('OK:', dst.name)


Ultralytics 8.4.161 🚀 Python-3.11.9 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 7806MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'YOLOv8n_HuggingFace_TomSmaildrone-yolo-v1.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (11.8 MB)

ONNX: starting export with onnx 1.23.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 1.4s, saved as 'YOLOv8n_HuggingFace_TomSmaildrone-yolo-v1.onnx' (11.7 MB)

TensorRT: starting export with TensorRT 11.3.0.99...
[10/01/2026-15:59:56] [TRT] [I] [MemUsageChange] Init CUDA: CPU +0, GPU +0, now: CPU 819, GPU 1761 (MiB)
[10/01/2026-15:59:56] [TRT] [I] ----------------------------------------------------------------
[10/01/2026-15:59:56] [TRT] [I] Input filename:   YOLOv8n_HuggingFace_TomSmaildrone-yolo-v1.onnx
[10/01/2026-15:59:56] [TRT] [I] ONNX IR version:  0.0.8
[10/01/2026-15:59:56] [TRT] [I] Opset version:    18
[10/01/202

In [24]:
# 3. TensorRT FP16 -> <model>_tensor_rt_16.engine
from pathlib import Path
from ultralytics import YOLO

SRC = Path('YOLOv8n_HuggingFace_TomSmaildrone-yolo-v1.pt')
IMGSZ = 640
DEVICE = 0
WORKSPACE = 0  # TensorRT workspace size flag; use integer MB-like value or leave unset

if ')_' in SRC.stem:
    prefix, rest = SRC.stem.split(')_', 1)
    prefix += ')'
    dst_name = f'{prefix}_tensor_rt_16_{rest}.engine'
else:
    dst_name = f'{SRC.stem}_tensor_rt_16.engine'

out = YOLO(str(SRC)).export(format='engine', half=True, imgsz=IMGSZ, device=DEVICE, workspace=WORKSPACE, verbose=False)
out = Path(out)
dst = out.parent / dst_name

if dst.exists():
    dst.unlink()

out.rename(dst)
print('OK:', dst.name)


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.161 🚀 Python-3.11.9 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 7806MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'YOLOv8n_HuggingFace_TomSmaildrone-yolo-v1.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (11.8 MB)

ONNX: starting export with onnx 1.23.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 0.6s, saved as 'YOLOv8n_HuggingFace_TomSmaildrone-yolo-v1.onnx' (11.7 MB)

TensorRT: starting export with TensorRT 11.3.0.99...
requirements: Ultralytics requirement ['nvidia-modelopt[onnx]>=0.44'] not found, attempting AutoUpdate...
Using Python 3.11.9 environment at: /home/daryna/miniconda3/envs/moca3d
Resolved 65 packages in 5.37s
   Building cppimport==26.4.17
      Built cppimport==26.4.17
 Downloaded nvidia-modelopt
 Downloaded pydantic-core
 Do

In [ ]:
# 4. TensorRT INT8 -> <model>_tensor_rt_8.engine
# Увага: INT8 потребує калібровки (CALIB_DATA). На чужому датасеті можлива втрата точності —
# для макс. якості підстав свій data.yaml з val-кадрами дронів.
from pathlib import Path
from ultralytics import YOLO

SRC = Path('YOLOv8n_HuggingFace_TomSmaildrone-yolo-v1.pt')
IMGSZ = 640
DEVICE = 0
WORKSPACE = 0  # TensorRT workspace size flag; use integer MB-like value or leave unset
CALIB_DATA = 'data.yaml'  # set to a valid calibration YAML on your target GPU machine

if not Path(CALIB_DATA).exists():
    raise FileNotFoundError(f'CALIB_DATA not found: {CALIB_DATA}. Set it to a valid calibration dataset YAML before INT8 export.')

if ')_' in SRC.stem:
    prefix, rest = SRC.stem.split(')_', 1)
    prefix += ')'
    dst_name = f'{prefix}_tensor_rt_8_{rest}.engine'
else:
    dst_name = f'{SRC.stem}_tensor_rt_8.engine'

out = YOLO(str(SRC)).export(format='engine', int8=True, data=CALIB_DATA, imgsz=IMGSZ, device=DEVICE, workspace=WORKSPACE, verbose=False)
out = Path(out)
dst = out.parent / dst_name

if dst.exists():
    dst.unlink()

out.rename(dst)
print('OK:', dst.name)


FileNotFoundError: CALIB_DATA not found: data.yaml. Set it to a valid calibration dataset YAML before INT8 export.

In [ ]:
# 5. Що вийшло + швидка перевірка завантаження
from pathlib import Path
from ultralytics import YOLO

for p in sorted(Path('.').glob('*_tensor_rt*.engine')):
    print(f'{p.name} — {p.stat().st_size / 1024**2:.1f} MB')

# Димовий тест (розкоментуй потрібний рядок):
# YOLO('1)_tensor_rt_YOLOv8n_Hugging_Face_TomSmaildrone-yolo-v1.engine').predict('https://ultralytics.com/images/bus.jpg')
# YOLO('1)_tensor_rt_16_YOLOv8n_Hugging_Face_TomSmaildrone-yolo-v1.engine').predict('https://ultralytics.com/images/bus.jpg')
# YOLO('1)_tensor_rt_8_YOLOv8n_Hugging_Face_TomSmaildrone-yolo-v1.engine').predict('https://ultralytics.com/images/bus.jpg')
